# POL: agente RL guiado por wallets on-chain

Este cuaderno cumple la secuencia de la guía de forma reproducible: **datos → perfiles → estado → transiciones → recompensas → Bellman → política → simulación**.

El agente opera cada **1 hora** y sólo usa wallets confirmadas como ganadoras para ese horizonte con `consistency_score ≥ 0.80`.

## 1. Problema y objetivo

Queremos decidir entre comprar POL, vender POL o mantener la posición. Las wallets ganadoras no sustituyen la rentabilidad del portafolio: aportan una señal adicional y causal sobre Buy/Sell/Hold. El objetivo es maximizar la riqueza acumulada neta de gas.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from pol_rl_pipeline import ejecutar_rl_desde_snapshot

# Usa el snapshot de perfiles ya generado. Cambia la ruta si quieres analizar otro.
snapshots = sorted(Path('informes_pol').glob('snapshot_*'))
if not snapshots:
    raise FileNotFoundError('Primero genera un informe de wallets en informes_pol/.')
SNAPSHOT_DIR = snapshots[-1]
OUTPUT_DIR = Path('informes_rl')

resultado = ejecutar_rl_desde_snapshot(
    snapshot_dir=SNAPSHOT_DIR,
    output_dir=OUTPUT_DIR,
    consistency_threshold=0.80,  # umbral oficial
    horizon=24,                 # 24 decisiones de una hora
    gamma=0.99,
)

print('Snapshot fuente:', SNAPSHOT_DIR)
print('Artefactos RL:', resultado.output_dir)

## 2. Cohorte que guía al agente

Una fila es `wallet × acción × 1h`. Sólo entra si ya era `winner` y su consistencia es al menos 0.80. `BUY`, `SELL` y `HOLD` son direcciones, no recomendaciones automáticas.

In [ ]:
wallets = resultado.wallets_dirigidas.copy()
columnas = [c for c in ['wallet', 'direccion_agente', 'consistency_score', 'n_decisiones', 'pnl_neto_usdc', 'retorno_neto_mediano'] if c in wallets]
display(wallets[columnas].head(15))
print('Wallets únicas dirigidas:', wallets['wallet'].nunique())

top = wallets.sort_values('consistency_score', ascending=False).head(15)
px.bar(
    top, x='consistency_score', y='wallet', color='direccion_agente', orientation='h',
    title='15 perfiles 1h más consistentes que pueden guiar al agente',
).update_yaxes(autorange='reversed').show()

## 3. MDP: estado, acciones y transiciones

El MDP es `M = (S, A, P, R, γ)`.

- Estado: `régimen de mercado × señal de wallets × posición`.
- Régimen: `DOWN`, `FLAT` o `UP`, construido con retornos pasados 1m, 5m, 15m y 1h; la banda lateral es fija para no usar futuro.
- Señal: la dirección con más soporte de wallets que ya habían madurado en ese corte; si no hay una dominante es `NEUTRAL`.
- Posición: `0 = USDC` o `1 = POL`.
- Acciones admisibles: desde USDC `{BUY_POL, HOLD}` y desde POL `{SELL_POL, HOLD}`.

In [ ]:
observaciones = resultado.observaciones.copy()
display(observaciones[['as_of', 'precio_t', 'precio_t1', 'regimen_mercado', 'senal_wallets', 'confianza_wallets']].head(10))

px.scatter(
    observaciones, x='as_of', y='confianza_wallets', color='senal_wallets',
    hover_data=['support_buy', 'support_sell', 'support_hold'],
    title='Señal causal de wallets disponible en cada decisión horaria',
).show()

Las probabilidades de transición se estiman contando cambios observados entre estados económicos consecutivos y aplicando suavizado de Laplace. La posición siguiente sí depende de la acción. Por tanto, cada fila válida de `P(s′|s,a)` debe sumar uno.

In [ ]:
from pol_rl_mdp import verificar_probabilidades

verificacion = verificar_probabilidades(resultado.mdp)
display(verificacion.head(12))
assert verificacion['valida'].all(), 'Hay una transición que no suma 1.'
print('Todas las transiciones de acciones admisibles suman 1.')

## 4. Recompensa

La recompensa base es el retorno de la posición que queda para la siguiente hora, menos gas si se compra o vende. No se da una recompensa fija por copiar una wallet. Sólo se suma un bonus moderado si la dirección dominante de las wallets coincide con una acción que realmente fue mejor que su alternativa: 

`R_final = R_base + 0.25 × confianza_wallets × ventaja_realizada`.

In [ ]:
from pol_rl_rewards import desglose_recompensas

fila = observaciones.iloc[0]
ejemplo = desglose_recompensas(
    posicion=0, precio_t=float(fila.precio_t), precio_t1=float(fila.precio_t1),
    costo_gas_ratio=float(fila.costo_gas_ratio),
    support_wallets={
        'support_buy': float(fila.support_buy),
        'support_sell': float(fila.support_sell),
        'support_hold': float(fila.support_hold),
    },
)
display(pd.DataFrame(ejemplo).T)

## 5. Bellman y política óptima

Se resuelve hacia atrás un horizonte finito de 24 decisiones: 

`V_t(s) = max_a [ R(s,a) + γ Σ P(s′|s,a)V_{t+1}(s′) ]`.

La tabla siguiente permite explicar qué acción toma la política para cada estado al inicio del episodio.

In [ ]:
politica = resultado.politica.copy()
display(politica.sort_values(['posicion', 'regimen_mercado', 'senal_wallets']))

px.scatter(
    politica, x='senal_wallets', y='regimen_mercado', color='accion_recomendada',
    symbol='accion_recomendada', facet_col='posicion', size='valor_optimo',
    title='Política Bellman: acción por estado (posición 0=USDC, 1=POL)',
).show()

## 6. Replay histórico explicable

Aplicamos la política sobre los precios de la misma ventana para revisar, hora por hora, la acción, recompensa base, bonus de wallets y riqueza acumulada. Es una demostración didáctica **in-sample**; no constituye una evaluación fuera de muestra.

In [ ]:
replay = resultado.replay.copy()
display(replay)
px.line(replay, x='as_of', y='wealth', markers=True, title='Riqueza del replay histórico (base inicial = 100)').show()
px.bar(replay, x='as_of', y=['recompensa_base', 'bonus_wallets'], barmode='relative', title='Descomposición de la recompensa por hora').show()

## 7. Limitaciones y siguiente paso

1. La muestra actual cubre 24 horas: es demasiado corta para declarar una política rentable o estable. Hace falta acumular snapshots y separar entrenamiento, validación y prueba temporal.
2. Las transiciones se estiman con pocas observaciones y se suavizan; representan una simulación pedagógica, no un mercado completo.
3. El modelo sólo representa posición long binaria y no infiere balances, transferencias externas ni impacto de mercado.

Próximo paso: almacenar ventanas sucesivas, fijar un corte temporal de entrenamiento y evaluar el replay fuera de muestra antes de pasar a un algoritmo RL de aprendizaje por interacción.